## Spaceship Titanic Prediction with Catboost

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold

## Import datasets

In [2]:
train = pd.read_csv("../input/spaceship-titanic/train.csv")
train_targets = train.pop("Transported")
test = pd.read_csv("../input/spaceship-titanic/test.csv")
data = pd.concat([train, test])
data["Cabin"] = data["Cabin"].replace(np.NAN, data["Cabin"].mode()[0])
data["Deck"] = data["Cabin"].apply(lambda item: str(item).split('/')[0])
data["Num"] = data["Cabin"].apply(lambda item:  str(item).split('/')[1])
data["Side"] = data["Cabin"].apply(lambda item: str(item).split('/')[2])
data.pop("Cabin")
data.pop("PassengerId")
data.pop("Name")
data = pd.get_dummies(data)
train = data.iloc[0:len(train)]
test = data.iloc[len(train):]


In [3]:
data.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,...,Num_992,Num_993,Num_994,Num_995,Num_996,Num_997,Num_998,Num_999,Side_P,Side_S
0,39.0,0.0,0.0,0.0,0.0,0.0,0,1,0,1,...,0,0,0,0,0,0,0,0,1,0
1,24.0,109.0,9.0,25.0,549.0,44.0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,1
2,58.0,43.0,3576.0,0.0,6715.0,49.0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,1
3,33.0,0.0,1283.0,371.0,3329.0,193.0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,1
4,16.0,303.0,70.0,151.0,565.0,2.0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,1


## Model Development

In [4]:
models = []
kfold = StratifiedKFold(7, shuffle=True, random_state=2022)
for index, (train_indices, valid_indices) in enumerate(kfold.split(train, train_targets)):
    x_train = train.iloc[train_indices]
    x_val = train.iloc[valid_indices]
    y_train = train_targets.iloc[train_indices]
    y_val = train_targets.iloc[valid_indices]
    params = {
        'iterations': 10000, 
        'depth': 8, 
        'early_stopping_rounds': 1000,
        'eval_metric': 'Accuracy',
        "verbose": 1000
    }
    ## Create Model
    model = CatBoostClassifier(**params)
    ## Train Model
    model.fit(x_train, y_train, eval_set=(x_val, y_val))
    ## Save Model
    file_path = f"model_{index}.model"
    model.save_model(file_path)
    ## Load Model
    model.load_model(file_path)
    models.append(model)

Learning rate set to 0.019114
0:	learn: 0.7727822	test: 0.7568438	best: 0.7568438 (0)	total: 74ms	remaining: 12m 19s
1000:	learn: 0.8476715	test: 0.7971014	best: 0.8003221 (416)	total: 20.8s	remaining: 3m 7s
Stopped by overfitting detector  (1000 iterations wait)

bestTest = 0.8003220612
bestIteration = 416

Shrink model to first 417 iterations.
Learning rate set to 0.019114
0:	learn: 0.7756006	test: 0.7842190	best: 0.7842190 (0)	total: 22.2ms	remaining: 3m 42s
1000:	learn: 0.8487451	test: 0.7995169	best: 0.8067633 (350)	total: 20.6s	remaining: 3m 5s
Stopped by overfitting detector  (1000 iterations wait)

bestTest = 0.806763285
bestIteration = 350

Shrink model to first 351 iterations.
Learning rate set to 0.019114
0:	learn: 0.7675480	test: 0.7504026	best: 0.7504026 (0)	total: 22.9ms	remaining: 3m 49s
1000:	learn: 0.8471346	test: 0.7987118	best: 0.8027375 (279)	total: 20.7s	remaining: 3m 6s
Stopped by overfitting detector  (1000 iterations wait)

bestTest = 0.8027375201
bestIteration 

## Feature Importance
Let's take a look at feature importance of this dataset.

In [5]:
for fold, model in enumerate(models):
    print("=" * 100)
    print(f"Feature Importance for fold {fold}:")
    print("=" * 100)
    feature_importance = sorted(zip(train.columns, model.get_feature_importance()), reverse=True, key=lambda item: item[1])
    for item in feature_importance[:30]:
        print(item)

Feature Importance for fold 0:
('Spa', 12.947715078704658)
('VRDeck', 11.727640713490132)
('RoomService', 8.882635531779911)
('CryoSleep_True', 8.029069074840656)
('FoodCourt', 7.937570822071622)
('HomePlanet_Earth', 7.056732349108351)
('HomePlanet_Europa', 4.955946764687835)
('Deck_E', 4.497611603207928)
('Age', 3.9002640412415492)
('ShoppingMall', 3.635486139239815)
('CryoSleep_False', 3.441248648225984)
('Deck_G', 3.2716731816443887)
('HomePlanet_Mars', 2.9795932983881985)
('Side_P', 2.808358556352005)
('Deck_C', 2.6942154011521637)
('Deck_F', 2.5194233235407446)
('Side_S', 2.1457925024584705)
('Destination_TRAPPIST-1e', 1.6973521588756977)
('Destination_55 Cancri e', 1.6510720992749024)
('Deck_B', 1.375760454500273)
('VIP_False', 0.39499109861390863)
('Deck_A', 0.3281605276658719)
('Destination_PSO J318.5-22', 0.2313365020912064)
('VIP_True', 0.17617721612680232)
('Deck_D', 0.14232648574378104)
('Num_981', 0.04904805596916021)
('Num_478', 0.030213095169718377)
('Num_307', 0.0263363

## Submission

In [6]:
def inference(df, models):
    y_pred = np.mean([model.predict_proba(df)[:, 1] for model in models], axis=0)
    y_pred = np.array(y_pred > 0.5, dtype=np.bool_)
    return y_pred

In [7]:
submission = pd.read_csv("../input/spaceship-titanic/sample_submission.csv")
submission["Transported"] = inference(test, models)
submission.to_csv("submission.csv", index=False)
submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
